In [2]:

from app.db_connector import get_engine
from app.models import Entity, SubTopic, SubTopic_Entity_Link, SubTopicDisplay
from app.transformers_util import generate_embeddings, get_util, get_model
from sqlalchemy.orm import Session
from sqlalchemy import (
    select,
    func,
    text
)
# import app.subtopics_util as subtopics_util
import json

engine = get_engine()
st_util = get_util()
st_model = get_model()

2024-04-15 10:57:20 - Connecting to database using TCP
2024-04-15 10:57:20 - Connected to database using TCP


In [4]:

user_id = "yU13Hk9BwEQiREgh91YM6EFKR7M2"
with Session(engine) as session:

    query = text(
            """SELECT s.id, s.name, s.description, 
                count(distinct d.id) as number_of_docs,
                json_agg(distinct d.id)
                FROM subtopic s
                JOIN subtopic_entity_link l ON l.subtopic_id = s.id
                JOIN entity e ON e.id = l.entity_id
                JOIN document d ON d.id = e.document_id
                WHERE s.user_id = '{USER_ID}'
            GROUP BY s.id, s.name, s.description
            ORDER BY count(distinct d.id) DESC""".format(USER_ID=user_id)
        )

    subtopic = session.execute(query).fetchall()

    print(SubTopicDisplay.from_touple(subtopic[0]))

id=73 name='Habit-Forming Product Development' description="The study and creation of products that form habits, led by Nir Eyal, author of the books 'Hooked: How to Build Habit-Forming Products' and 'Indistractable: How to Control Your Attention and Choose Your Life'." number_of_docs=4 docs_ids=[133, 134, 135, 136]


In [14]:
clusters = subtopics_util.generate_clusters(sentences)

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

In [15]:
subtopics = []
for ci, cluster in enumerate(clusters):
    print(f"cluster {ci}")
    subtopic = SubTopic(name=f"subtopic_{ci}")
    for i in cluster:
        entity = entities[i]
        print(f"{entity.id}: {entity.name} - {entity.description}")
    

cluster 0
923: Sacramento - The capital city of California, where Marquese Chriss grew up and played football and basketball
995: Sacramento - The capital city of California, where Marquese Chriss grew up and played football and basketball
681: Sacramento - The capital city of California, where Marquese Chriss grew up and played football and basketball
947: Sacramento - The capital city of California, where Marquese Chriss grew up and played football and basketball
935: Sacramento - The capital city of California, where Marquese Chriss grew up and played football and basketball
1019: Sacramento - The capital city of California, where Marquese Chriss grew up and played football and basketball
983: Sacramento - The capital city of California, where Marquese Chriss grew up and played football and basketball
959: Sacramento - The capital city of California, where Marquese Chriss grew up and played football and basketball
1007: Sacramento - The capital city of California, where Marquese Chr

In [16]:
subtopics = []
for ci, cluster in enumerate(clusters):
    subtopic = await subtopics_util.create_subtopic(cluster, entities)
    subtopics.append(subtopic)
    print(f"subtopic {ci}: {subtopic.name}")
    

2024-04-12 14:06:09 - Attempt 1 to generate summary
2024-04-12 14:06:12 - Response status: finished
2024-04-12 14:06:12 - Answer is not None. Stop retrying. Number of attempts 1


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

subtopic 0: Marquese Chriss's Background
2024-04-12 14:06:12 - Attempt 1 to generate summary
2024-04-12 14:06:14 - Response status: finished
2024-04-12 14:06:14 - Answer is not None. Stop retrying. Number of attempts 1


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

subtopic 1: Retrieval-Augmented Generation (RAG) Technology
2024-04-12 14:06:15 - Attempt 1 to generate summary
2024-04-12 14:06:17 - Response status: finished
2024-04-12 14:06:17 - Answer is not None. Stop retrying. Number of attempts 1


...

subtopic 6: National Basketball Association
2024-04-12 14:06:28 - Attempt 1 to generate summary
2024-04-12 14:06:33 - Response status: finished
2024-04-12 14:06:33 - Answer is not None. Stop retrying. Number of attempts 1


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

subtopic 7: Video Conferencing Tools
2024-04-12 14:06:33 - Attempt 1 to generate summary
2024-04-12 14:06:36 - Response status: finished
2024-04-12 14:06:36 - Answer is not None. Stop retrying. Number of attempts 1


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

subtopic 8: Email Marketing Strategy


In [6]:
print(len(subtopics))
print(len(clusters))

simiarity_compairs = []
for i, cluster in enumerate(clusters):
    subtopic = subtopics[i]
    print(f"subtopic {i}: {subtopic.name}")
    for j in cluster:
        entity = entities[j]
        cosine_similarity = st_util.cos_sim(entity.embedding, subtopic.embedding)
        print(f"Entity: {entity.name} - {entity.description} -> similarity: {cosine_similarity}")
    print("\n\n")

20
20
subtopic 0: Detroit Pistons Players
Entity: Rasheed Wallace - Power forward for the Detroit Pistons from 2004-2009, known for his defense and shooting abilities -> similarity: tensor([[0.3769]])
Entity: Ben Wallace - Center for the Detroit Pistons from 2000-2006 and 2009-2012, known for his defense and rebounding abilities -> similarity: tensor([[0.3901]])
Entity: Josh Smith - Power forward for the Detroit Pistons from 2013-2015, known for his inefficiency and poor defense -> similarity: tensor([[0.3094]])
Entity: Rip Hamilton - Shooting guard for the Detroit Pistons from 2002-2011, known for his scoring abilities -> similarity: tensor([[0.3608]])
Entity: Derrick Williams - Power forward for the Sacramento Kings from 2015-2016, known for his scoring abilities -> similarity: tensor([[0.2329]])
Entity: Andre Drummond - Center for the Detroit Pistons from 2012-2021, known for his rebounding and shot-blocking abilities -> similarity: tensor([[0.3752]])
Entity: Kentavious Caldwell-Pop

In [22]:
for i, cluster in enumerate(clusters):
    subtopic = subtopics[i]
    print(f"subtopic {i}: {subtopic.name}")
    for j in cluster:
        entity = entities[j]
        print(f"{entity.id}")
    print("\n\n")

subtopic 0: Detroit Pistons Players
749
746
738
748
755
737
753
744
751
745
739
752
750
747
736
754
740
743
757
756
759
758
741
686
819
675



subtopic 1: Video Conferencing Platforms
720
732
712
731
726
716
719
710
700
706
705
694



subtopic 2: Nir Eyal's Books and Teachings
702
729
718
722
709
703
730
723
727



subtopic 3: Product Management
858
840
841
838
857
859
802
665



subtopic 4: Marketing Practices
801
804
762
782
789
836
806



subtopic 5: Vector Search and Databases
653
656
655
697
698



subtopic 6: NBA Draft Process
684
685
677
663



subtopic 7: Distraction-Free Meetings
734
701
717
707



subtopic 8: Socratic Method
713
724
733
704



subtopic 9: Hoka Acquisition by Deckers
820
821
823
824



subtopic 10: Detroit Pistons
735
742
689



subtopic 11: National Basketball Association
676
815
661



subtopic 12: Leadership Concepts
664
670
674



subtopic 13: Cold Email Outreach
780
760
800



subtopic 14: Basketball Career of Marquese Chriss
682
681
688



subtopic 15: D

In [19]:
len(entities)

156

In [8]:
from app.models import Entity
from sqlalchemy.orm import Session
## Group one of entities
group_one_clousters_names = ['Retrieval-Augmented Generation (RAG)', 'Basketball Career of Marquese Chriss', 
                         'Vector Search and Databases', "Nir Eyal's Books and Teachings"]
group_one = [639, 648, 682, 681, 653, 656, 655, 702, 729, 718, 722, 709]


### Group two of entities. Same, groupd names as group one
group_two = [690, 688, 697, 698, 703, 730, 723, 727]

group_three_clusters_names = ["Video Conferencing Platforms", "Detroit Pistons Players"]
group_three = [712, 731, 726, 716, 719, 749, 746, 738, 748, 755, 737, 753]


group_three_str = []
for index in group_three:
    
    for entity in entities:
        if entity.id == index:
            et = Entity(name=entity.name, type=entity.type, description=entity.description, document_id=entity.document_id)
            group_three_str.append(et.model_dump_json())

print(group_three_str)


['{"document_id":136,"name":"Zoom","description":"A company that provides video conferencing services","type":"company","id":null,"source":null,"wikidata_id":null,"score":null}', '{"document_id":136,"name":"virtual meetings","description":"Meetings held through video conferencing technology","type":"topic","id":null,"source":null,"wikidata_id":null,"score":null}']


In [51]:
for se in group_one_str:
    entity = Entity(**json.loads(se))
    print(entity.name)

retrieval-augmented generation (RAG)
Retrieval-Augmented Generation (RAG)
University of Washington
Sacramento
Vector search retrieval methods
Vector indexes
Hybrid retrieval approach
Nir Eyal
Hooked
Nir Eyal
Hooked: How to Build Habit-Forming Products
Nir Eyal


In [10]:
from sqlalchemy import and_, text
from sqlalchemy import select
from app.models import Bookmark, Document, Entity, SubTopic_Entity_Link
from sqlalchemy.orm import Session
from sqlalchemy import (
    select,
    func,
)
from app.db_connector import get_engine
engine = get_engine()

with Session(engine) as session:

        filter_stmt = select(Entity.id).join(Document, Document.id == Entity.document_id)\
            .join(Bookmark, Document.id == Bookmark.document_id)\
            .where(Bookmark.user_id == 'yU13Hk9BwEQiREgh91YM6EFKR7M2')
        
        
        stmt = select(Entity).outerjoin(SubTopic_Entity_Link, Entity.id == SubTopic_Entity_Link.entity_id)\
            .where(and_(
                SubTopic_Entity_Link.subtopic_id == None,
                Entity.id.in_(session.execute(filter_stmt).scalars().unique().all())
                ))
        
        entities = session.scalars(stmt).unique().all() 
        print(type(entities[0]) == Entity)


2024-04-12 13:44:56 - Connecting to database using TCP
2024-04-12 13:44:56 - Connected to database using TCP
True
